# Lab 07: Latent Spaces

Uses `lab07_starter.py` unchanged. 8 by 8 digits, 1,438 training and 359 test images, pixels in [0, 1]. Every model has an 8-dimensional code and one hidden layer of 64 units on each side, and trains for 200 epochs at 2e-3 with batch 64 on seeds 0, 1 and 2 unless noted.

In [1]:
import json
import os
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import lab07_starter as S

SEEDS = [0, 1, 2]
LATENT_DIM = 8
HIDDEN_DIM = 64
EPOCHS, LEARNING_RATE, BATCH_SIZE = 200, 2e-3, 64
N_SAMPLES = 359  # every Frechet distance in this notebook uses 359 images
os.makedirs("figures", exist_ok=True)
results = {}
notebook_start = time.perf_counter()

## Run the starter once

Before writing anything I want to know the environment is fine, so I run the starter file exactly as it was handed out. A pass is the script finishing and printing a reconstruction error and a set of sample scores.

In [2]:
starter_start = time.perf_counter()
starter_run = subprocess.run([sys.executable, "lab07_starter.py"], capture_output=True, text=True, check=True)
print(starter_run.stdout)
print(f"starter took {time.perf_counter() - starter_start:.1f}s")

train (1438, 64) test (359, 64)
autoencoder test reconstruction MSE: 0.0163
feature network accuracy on real test images: 0.972
autoencoder samples from N(0, I): {'frechet': 86.242, 'precision': 0.437, 'recall': 0.794, 'nearest_train': 2.033}
saved starter_check.png

starter took 10.4s


It finished in well under a minute. The feature network gets 0.972 on the real test images, which is what every sample score below leans on.

## Load the data and train the one feature network

Every Fréchet distance, precision and recall in this lab has to come from the same feature network, otherwise the numbers can't be compared. I train it once here with the starter's defaults (seed 0) and reuse `feature_net` and `test_features` for every score.

In [3]:
Xtr, ytr, Xte, yte = S.load_task()
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))
print(f"pixel range: {Xtr.min().item():.2f} to {Xtr.max().item():.2f}")
print("train images per digit:", torch.bincount(ytr, minlength=10).tolist())
print("test images per digit: ", torch.bincount(yte, minlength=10).tolist())

feature_net = S.train_feature_net(Xtr, ytr)
test_features = S.get_features(feature_net, Xte)
with torch.no_grad():
    feature_net_accuracy = (feature_net(Xte).argmax(1) == yte).float().mean().item()
print(f"feature network test accuracy: {feature_net_accuracy:.3f}   features per image: {test_features.shape[1]}")
results["feature_net_accuracy"] = feature_net_accuracy

train (1438, 64) test (359, 64)
pixel range: 0.00 to 1.00
train images per digit: [151, 147, 141, 154, 151, 142, 137, 140, 136, 139]
test images per digit:  [27, 35, 36, 29, 30, 40, 44, 39, 38, 41]


feature network test accuracy: 0.972   features per image: 32


Same 0.972 as the starter's own run, so `train_feature_net` is deterministic and the network I score with is the one the starter checked.